<h1 align="center"> MNIST Handwritten Digit Classification (Multi Class Classification)</h1>

<img 
    src="https://www.kaggle.com/competitions/3004/images/header"
    alt="" 
    width="300" 
    height="200" 
    style="display: block; margin: 0 auto; border-radius:15px" 
/>

- MNIST ("Modified National Institute of Standards and Technology") is the de facto “hello world” dataset of computer vision. Since its release in 1999, this classic dataset of handwritten images has served as the basis for benchmarking classification algorithms.

- Image classification is a fundamental task in deep learning. It introduces concepts like feature extraction, convolutional neural networks (CNNs), and overfitting.
  
- MNIST is a dataset of 70,000 grayscale images of handwritten digits (0 to 9).

- Each image is 28x28 pixels.

- Goal: Build a model to correctly classify these images into the correct digit.


## Basics of Neural Networks & CNNs

### What is a Neural Network?

- Inspired by the brain’s neurons, a neural network consists of layers of nodes (“neurons”).
- Each neuron applies a weighted sum of inputs + bias and passes it through an activation function.

### What is a Convolutional Neural Network (CNN)?
- CNNs are designed to process images.
- They use convolutional layers to detect patterns like edges, shapes, and textures.
- CNN layers preserve spatial relationships, unlike fully connected layers.

### Key CNN components:

1. Convolutional layers: Apply filters/kernels sliding over the image to create feature maps.
2. Pooling layers: Reduce spatial size (e.g., max pooling) to lower computation and control overfitting.
3. Fully connected layers: At the end, they combine extracted features to classify.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%config InlineBackend.figure_format='svg'

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

tf.config.list_physical_devices('GPU')


---

# Step 1: Exploratory Data Analysis (EDA)

---

In [ ]:
train_data = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test_data = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

print(f"Train data null count: {train_data.isnull().sum().sum()}")
print(f"Train data null count: {test_data.isnull().sum().sum()}")


In [ ]:
train_data.head()

In [ ]:
# Pick a sample
sample = train_data.iloc[0, 1:].values.reshape(28, 28)

plt.figure(figsize=(3,3))
plt.imshow(sample, cmap='gray')
plt.title(f"Label: {train_data.iloc[0, 0]}")
plt.axis('off')
plt.show()


In [ ]:
# Set grid size
rows, cols = 5, 5
fig, axes = plt.subplots(rows, cols, figsize=(10,10))

# Flattened pixel data and labels
images = train_data.drop("label", axis=1).values
labels = train_data["label"].values

# Loop through grid
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]
    ax.imshow(images[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {labels[i]}")
    ax.axis('off')

plt.tight_layout()
plt.show()


---

# Step 2: Data Preprocessing

---


In [ ]:
# Separate features and labels
X = train_data.drop("label", axis=1).values
y = train_data["label"].values

# Normalize pixel values
X = X / 255.0
test_X = test_data.values / 255.0

# Reshape for CNN input
X = X.reshape(-1, 28, 28, 1)
test_X = test_X.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_cat = to_categorical(y)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y_cat, test_size=0.1, random_state=42)


---

# Step 3: Model Training & Evaluation

---

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val)
)

In [ ]:
# Plot training history
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.show()


---

# Step 4: Test Data Prediction¶

---

In [ ]:
predictions = model.predict(test_X)
predicted_labels = np.argmax(predictions, axis=1)


In [ ]:
submission = pd.read_csv("/kaggle/input/digit-recognizer/sample_submission.csv")
submission.head()

In [ ]:
submission["Label"] = predicted_labels
submission.to_csv("submission.csv", index=False)
submission.head()